# Phase3. Single Factor Testing

## 3.1 Loading Data & Packages

In [102]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [103]:
from pathlib import Path
import json
import pandas as pd

data_dir = Path('data/alpha_inputs')
manifest_path = data_dir / 'manifest.json'
manifest = json.loads(manifest_path.read_text())
fmt = manifest.get('format')
files = manifest.get('files', {})

dataframes = {}
for name, path_str in files.items():
    path = Path(path_str)
    if fmt == 'parquet':
        dataframes[name] = pd.read_parquet(path)
    elif fmt == 'pickle':
        dataframes[name] = pd.read_pickle(path)
    else:
        raise ValueError(f'Unknown data format: {fmt}')

globals().update(dataframes)


In [104]:
industry = pd.read_csv('/Users/apple/Desktop/PitchBook/Multi-Factor L:S/stoxx50_ind.csv')
industries = (
    industry
    .set_index("Instrument")["TRBC Industry Group Name"]   # or "TRBC Industry Group Code"
)

industries = industries.reindex(close_df.columns)

In [105]:
subindustries = (
    industry
    .set_index("Instrument")["TRBC Industry Name"]   # or "TRBC Industry Group Code"
)
subindustries = subindustries.reindex(close_df.columns)

In [106]:
pip install nbformat

Note: you may need to restart the kernel to use updated packages.


In [107]:
%run "./Factors.ipynb"

Note: you may need to restart the kernel to use updated packages.


## 3.2 Alpha Metrics

In [108]:
# Print Performance Metrics
def performance_summary(pnl: pd.Series, periods_per_year: int = 252):
    pnl = pnl.dropna()
    # Skip the Training Period
    non_zero = pnl != 0
    if non_zero.any():
        first_idx = pnl[non_zero].index[0]
        pnl = pnl.loc[first_idx:]

    n = len(pnl)

    # cumulative equity (start at 1.0)
    equity = (1 + pnl).cumprod()

    # total return
    total_return = equity.iloc[-1] - 1.0

    # CAGR
    years = n / periods_per_year
    if years > 0:
        cagr = equity.iloc[-1] ** (1 / years) - 1
    else:
        cagr = np.nan

    # annualized volatility
    ann_vol = pnl.std(ddof=1) * np.sqrt(periods_per_year)

    # Sharpe
    sharpe = cagr / ann_vol if pd.notna(ann_vol) and ann_vol != 0 else np.nan

    # drawdowns
    running_max = equity.cummax()
    drawdown = equity / running_max - 1.0
    max_dd = drawdown.min()

    # Calmar
    calmar = cagr / abs(max_dd) if pd.notna(max_dd) and max_dd < 0 else np.nan


    # hit rate
    hit_rate = (pnl > 0).mean()

    print("===== Performance Summary =====")
    print(f"Periods             : {n}")
    print(f"Total Return        : {total_return:8.2%}")
    print(f"CAGR                : {cagr:8.2%}")
    print(f"Ann. Volatility     : {ann_vol:8.2%}")
    print(f"Sharpe Ratio        : {sharpe:8.2f}")
    print(f"Max Drawdown        : {max_dd:8.2%}")
    print(f"Calmar Ratio        : {calmar:8.2f}")
    print(f"Hit Rate (p>0)      : {hit_rate:8.2%}")

In [109]:
# Plot Equity Backtest Curve
def plot_equity_curve(pnl: pd.Series, title: str = "Equity Curve"):
    pnl = pnl.replace([np.inf, -np.inf], np.nan).dropna()

    # trim leading zeros
    non_zero = pnl != 0
    if non_zero.any():
        first_idx = pnl[non_zero].index[0]
        pnl = pnl.loc[first_idx:]
        
    equity = (1 + pnl).cumprod()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(equity.index, equity.values)
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Equity (start = 1.0)")
    ax.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.show()

## 3.3 Data Pre-Processors

### 3.3.1 Outlier Clipper

In [110]:
def clip_outliers(df, n_std=3.0):
    """
    For each timestamp (row), clip values to mean ± n_std * std.
    """
    out = df.copy().astype(float)
    for time in out.index:
        row = out.loc[time].values.astype(float)
        mask = np.isfinite(row)
        if mask.sum() == 0:
            continue
        vals = row[mask]
        mu = vals.mean()
        sigma = vals.std(ddof=1)
        if sigma == 0 or not np.isfinite(sigma):
            continue
        lo = mu - n_std * sigma
        hi = mu + n_std * sigma
        out.loc[time] = np.clip(row, lo, hi)
    return out

### 3.3.2 Standardization

In [111]:
def standardize(df):
    out = df.copy().astype(float)
    for time in out.index:
        row = out.loc[time].values.astype(float)
        mask = np.isfinite(row)
        if mask.sum() == 0:
            continue
        vals = row[mask]
        mu = vals.mean()
        sigma = vals.std(ddof=1)
        if sigma == 0 or not np.isfinite(sigma):
            out.loc[time, mask] = 0.0
            continue
        z = (row - mu) / sigma
        out.loc[time] = z
    return out


### 3.3.3 Fundamental data processing

In [112]:
def expand_fundamental_to_daily(fund_df, daily_index, lag_days=45):
    """
    fund_df: index = report_date (or fiscal end date), columns = tickers
    daily_index: full daily index you use for prices
    lag_days: conservative lag until info is usable (e.g. 45 days)
    """
    fund = fund_df.copy().astype(float)
    fund.index = pd.to_datetime(fund.index)
    fund = fund.sort_index()

    if not fund.index.is_unique:
        fund = fund.groupby(level=0).last()

    fund.index = fund.index + pd.Timedelta(days=lag_days)

    # 2) reindex to daily calendar, forward-fill (never backward-fill!)
    daily = fund.reindex(daily_index).ffill()

    return daily

In [113]:
liabilities_df = expand_fundamental_to_daily(liabilities_df,daily_index=close_df.index)
equity_df = expand_fundamental_to_daily(equity_df,daily_index=close_df.index)
assets_df = expand_fundamental_to_daily(assets_df,daily_index=close_df.index)
debt_df = expand_fundamental_to_daily(debt_df,daily_index=close_df.index)
operating_income_df = expand_fundamental_to_daily(operating_income_df,daily_index=close_df.index)

## 3.4 Analytics

In [114]:
def factor_forward_returns_wide(alpha_df, close_df, periods=(1, 5, 10), min_assets=5):
    """
    Use original wide format (index=dates, columns=assets).
    Returns:
      - factor_aligned (DataFrame)
      - fwd_returns (dict of DataFrames)
      - ic (DataFrame: index=dates, columns=periods)
    """
    # align dates/assets
    common_dates = alpha_df.index.intersection(close_df.index)
    common_assets = alpha_df.columns.intersection(close_df.columns)
    factor = alpha_df.loc[common_dates, common_assets].astype(float)
    prices = close_df.loc[common_dates, common_assets].astype(float)

    # forward returns
    fwd_returns = {}
    for p in periods:
        fwd = prices.pct_change(p).shift(-p)
        # mask where factor is missing
        fwd = fwd.where(factor.notna())
        fwd_returns[p] = fwd

    # IC per date (Spearman)
    ic = pd.DataFrame(index=factor.index, columns=list(periods), dtype=float)
    for p, fwd in fwd_returns.items():
        for dt in factor.index:
            x = factor.loc[dt]
            y = fwd.loc[dt]
            mask = x.notna() & y.notna()
            if mask.sum() < min_assets:
                ic.at[dt, p] = np.nan
                continue
            ic.at[dt, p] = x[mask].rank().corr(y[mask].rank())

    return factor, fwd_returns, ic

## 3.5 Factor Implementation

## 0 Alpha NLTSMOM

In [115]:
fwd_ret = close_df.shift(-1) / close_df - 1.0

In [116]:
def make_weights(alpha_row):
    "Convert 1D alpha (for one date) into long-short weights."
    a = alpha_row.copy()
    a = a.replace([np.inf, -np.inf], np.nan)
    a = a.fillna(0.0)

    a = a - a.mean()
    # scale so sum(|w|) = 1
    denom = a.abs().sum()    
    return a / denom

In [117]:
alpha_nltsmom = alpha_nltsmom_ann(close_df)

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_27716/3468435781.py:28: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = close.pct_change()


In [118]:
alpha_panel0 = alpha_nltsmom
alpha_panel0 = alpha_panel0.iloc[:-1, :]

# Apply weights and compute pnl
weights0 = alpha_panel0.apply(make_weights, axis=1)
daily_pnl0 = (weights0 * fwd_ret).sum(axis=1)

## 1 Alpha 0617a

In [119]:
alpha1 = alpha_0617a(open_df,volume_df)

In [120]:
factor, fwd, ic = factor_forward_returns_wide(alpha1, close_df, periods=(1,5,10))

/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_27716/2569342948.py:18: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_27716/2569342948.py:18: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  fwd = prices.pct_change(p).shift(-p)
/var/folders/n7/k_8ylm196lz3lzb72_x8vz340000gn/T/ipykernel_27716/2569342948.py:18: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling

In [121]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel = alpha1
alpha_panel = alpha_panel.iloc[:-1, :]
fwd_ret     = fwd_ret.iloc[:-1, :]

# Apply weights and compute pnl
weights = alpha_panel.apply(make_weights, axis=1)  # same shape as alpha_panel
daily_pnl = (weights * fwd_ret).sum(axis=1)

## 2 Alpha 0810a

In [122]:
alpha2 = alpha_0810a(returns_df,cap_df,volume_df)

In [123]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel2 = alpha2
alpha_panel2 = alpha_panel2.iloc[:-1, :]

# Apply weights and compute pnl
weights2 = alpha_panel2.apply(make_weights, axis=1)
daily_pnl2 = (weights2 * fwd_ret).sum(axis=1)

## 3 Alpha 0415b

In [124]:
alpha3 = alpha_0415b(close_df,volume_df)

In [125]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel3 = alpha3
alpha_panel3 = alpha_panel3.iloc[:-1, :]

# Apply weights and compute pnl
weights3 = alpha_panel3.apply(make_weights, axis=1)
daily_pnl3 = (weights3 * fwd_ret).sum(axis=1)

## 4 Alpha 0616a

In [126]:
alpha4 = alpha_0616a(close_df,open_df,volume_df)

In [127]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel4 = alpha4
alpha_panel4 = alpha_panel4.iloc[:-1, :]

# Apply weights and compute pnl
weights4 = alpha_panel4.apply(make_weights, axis=1)
daily_pnl4 = (weights4 * fwd_ret).sum(axis=1)

## 5 Alpha0812a

In [128]:
alpha5 = alpha_0812a(liabilities_df,assets_df,debt_df,equity_df)

In [129]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel5 = alpha5
alpha_panel5 = alpha_panel5.iloc[:-1, :]

# Apply weights and compute pnl
weights5 = alpha_panel5.apply(make_weights, axis=1)
daily_pnl5 = (weights5 * fwd_ret).sum(axis=1)

## 6 Alpha0612d

In [130]:
alpha6 = alpha_0612d(debt_df,assets_df,volume_df)

/Users/apple/Documents/GitHub/Quant/notebooks/operators_library.py:140: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cond = cond_df.reindex(val_df.index).fillna(False).astype(bool)


In [131]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel6 = alpha6
alpha_panel6 = alpha_panel6.iloc[:-1, :]

# Apply weights and compute pnl
weights6 = alpha_panel6.apply(make_weights, axis=1)
daily_pnl6 = (weights6 * fwd_ret).sum(axis=1)

## 7 Alpha 0608d

In [132]:
alpha7 = alpha_0608d(close_df,open_df,volume_df,sharesout_df,industries,horro_window=10)

In [133]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel7 = alpha7
alpha_panel7 = alpha_panel7.iloc[:-1, :]

# Apply weights and compute pnl
weights7 = alpha_panel7.apply(make_weights, axis=1)
daily_pnl7 = (weights7 * fwd_ret).sum(axis=1)

## 8 Alpha0615a

In [134]:
alpha8 = alpha_0615a(volume_df,sharesout_df,cap_df)

In [135]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel8 = alpha8
alpha_panel8 = alpha_panel8.iloc[:-1, :]

# Apply weights and compute pnl
weights8 = alpha_panel8.apply(make_weights, axis=1)
daily_pnl8 = (weights8 * fwd_ret).sum(axis=1)

## 9 Alpha0604c

In [136]:
alpha9 = alpha_0604c(close_df,high_df,low_df)

In [137]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel9 = alpha9
alpha_panel9 = alpha_panel9.iloc[:-1, :]

# Apply weights and compute pnl
weights9 = alpha_panel9.apply(make_weights, axis=1)
daily_pnl9 = (weights9 * fwd_ret).sum(axis=1)

## 10 Alpha0407a

In [138]:
alpha10 = alpha_0407a(debt_df)

In [139]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel10 = alpha10
alpha_panel10 = alpha_panel10.iloc[:-1, :]

# Apply weights and compute pnl
weights10 = alpha_panel10.apply(make_weights, axis=1)
daily_pnl10 = (weights10 * fwd_ret).sum(axis=1)

## 11 Alpha0505

In [140]:
alpha11 = alpha_0505(operating_income_df,vwap_df)

In [141]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel11 = alpha11
alpha_panel11 = alpha_panel11.iloc[:-1, :]

# Apply weights and compute pnl
weights11 = alpha_panel11.apply(make_weights, axis=1)
daily_pnl11 = (weights11 * fwd_ret).sum(axis=1)

## 12 Alpha0413a

In [143]:
alpha12 = alpha_0413a(close_df,high_df,low_df)

In [144]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel12 = alpha12
alpha_panel12 = alpha_panel12.iloc[:-1, :]

# Apply weights and compute pnl
weights12 = alpha_panel12.apply(make_weights, axis=1)
daily_pnl12 = (weights12 * fwd_ret).sum(axis=1)

## 13 Alpha0413b

In [145]:
alpha13 = alpha_0413b(operating_income_df,cap_df)

In [146]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel13 = alpha13
alpha_panel13 = alpha_panel13.iloc[:-1, :]

# Apply weights and compute pnl
weights13 = alpha_panel13.apply(make_weights, axis=1)
daily_pnl13 = (weights13 * fwd_ret).sum(axis=1)

## 14 Alpha0503b

## 15 Alpha0603a

In [147]:
alpha15 = alpha_0603a(volume_df,sharesout_df,cap_df)

In [148]:
# Backtest
# align shapes: drop last row (no forward return)
alpha_panel15 = alpha15
alpha_panel15 = alpha_panel15.iloc[:-1, :]

# Apply weights and compute pnl
weights15 = alpha_panel15.apply(make_weights, axis=1)
daily_pnl15 = ((weights15 * fwd_ret).sum(axis=1).fillna(0.0))

## 16 Alpha0416a

In [149]:
alpha16 = alpha_0416a(returns_df,subindustries,volume_df)

In [150]:
alpha_panel16 = alpha16
alpha_panel16 = alpha_panel16.iloc[:-1, :]

# Apply weights and compute pnl
weights16 = alpha_panel16.apply(make_weights, axis=1)
daily_pnl16 = ((weights16 * fwd_ret).sum(axis=1).fillna(0.0))

## 17 Alpha0421b

In [151]:
alpha17 = alpha_0421b(close_df,open_df,volume_df,returns_df)

/Users/apple/Documents/GitHub/Quant/notebooks/operators_library.py:140: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cond = cond_df.reindex(val_df.index).fillna(False).astype(bool)


In [152]:
alpha_panel17 = alpha17
alpha_panel17 = alpha_panel17.iloc[:-1, :]

# Apply weights and compute pnl
weights17 = alpha_panel17.apply(make_weights, axis=1)
daily_pnl17 = ((weights17 * fwd_ret).sum(axis=1).fillna(0.0))

## 18 Alpha0403c

In [153]:
alpha18 = alpha_0403c(volume_df,sharesout_df)

In [154]:
alpha_panel18 = alpha18
alpha_panel18 = alpha_panel18.iloc[:-1, :]

# Apply weights and compute pnl
weights18 = alpha_panel18.apply(make_weights, axis=1)
daily_pnl18 = ((weights18 * fwd_ret).sum(axis=1).fillna(0.0))

## 19 Alpha0618e

In [155]:
alpha19 = alpha_0618e(vwap_df,close_df,volume_df)

In [156]:
alpha_panel19 = alpha19
alpha_panel19 = alpha_panel19.iloc[:-1, :]

# Apply weights and compute pnl
weights19 = alpha_panel19.apply(make_weights, axis=1)
daily_pnl19 = ((weights19 * fwd_ret).sum(axis=1).fillna(0.0))

## 20 Alpha0611

## 21 Alpha0404b

In [157]:
alpha21 = alpha_0404b(returns_df,operating_income_df,cap_df)

/Users/apple/Documents/GitHub/Quant/notebooks/operators_library.py:140: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cond = cond_df.reindex(val_df.index).fillna(False).astype(bool)


In [158]:
alpha_panel21 = alpha21
alpha_panel21 = alpha_panel21.iloc[:-1, :]

# Apply weights and compute pnl
weights21 = alpha_panel21.apply(make_weights, axis=1)
daily_pnl21 = ((weights21 * fwd_ret).sum(axis=1).fillna(0.0))

## 22 Alpha0412

In [159]:
alpha22 = alpha_0412(operating_income_df,vwap_df,volume_df,returns_df)

In [160]:
alpha_panel22 = alpha22
alpha_panel22 = alpha_panel22.iloc[:-1, :]

# Apply weights and compute pnl
weights22 = alpha_panel22.apply(make_weights, axis=1)
daily_pnl22 = ((weights22 * fwd_ret).sum(axis=1).fillna(0.0))

## 23 Alpha0421a

## 24 Alpha0418b

In [161]:
alpha24 = alpha_0418b(operating_income_df,vwap_df)

In [162]:
alpha_panel24 = alpha24
alpha_panel24 = alpha_panel24.iloc[:-1, :]

# Apply weights and compute pnl
weights24 = alpha_panel24.apply(make_weights, axis=1)
daily_pnl24 = ((weights24 * fwd_ret).sum(axis=1).fillna(0.0))